# Topic: Cryptographic Defense Engineering - AES-CBC vs AES-GCM modes, PKCS7 block padding, and RSA key size validations
 
## 1. SYMMETRIC BLOCK CIPHERS & PADDING
- **Block Ciphers**: Algorithms like AES process data in fixed-size blocks (128 bits / 16 bytes).
- **PKCS7 Padding**:
  - If plaintext is not a multiple of 16 bytes, we must pad the final block.
  - **Padding Rule**: Append $N$ bytes, where each byte contains the value of $N$. If the data is already aligned, append a full block of 16 bytes of value `0x10` (16) to ensure unambiguous unpadding.
- **AES-CBC (Cipher Block Chaining)**:
  - Chains blocks together: $C_i = E_K(P_i \oplus C_{i-1})$.
  - Requires an Initialization Vector (IV).
  - **Vulnerability**: Does not verify data integrity. Subject to **Padding Oracle Attacks** or ciphertext tampering.
- **AES-GCM (Galois/Counter Mode)**:
  - Authenticated encryption (AEAD). Excludes padding overhead by acting as a stream cipher under the hood and appends an integrity validation tag.
 
## 2. ASYMMETRIC KEY SIZE VALIDATIONS
- **RSA Safety**:
  - Production standards dictate **minimum 2048-bit RSA keys**.
  - Keys under 1024 bits are mathematically vulnerable to factorization attacks using modern distributed computing.
 
---


In [ ]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding

class SymmetricCBCModule:
    """Implements AES-CBC encryption with PKCS7 padding."""
    def __init__(self, key_bytes):
        assert len(key_bytes) in (16, 24, 32), "Key must be 128, 192, or 256 bits"
        self.key = key_bytes

    def encrypt(self, plaintext_bytes):
        iv = os.urandom(16)  # AES block size is 16 bytes
        
        # Apply PKCS7 padding
        padder = padding.PKCS7(128).padder()
        padded_data = padder.update(plaintext_bytes) + padder.finalize()
        
        # Instantiate CBC Cipher
        cipher = Cipher(algorithms.AES(self.key), modes.CBC(iv))
        encryptor = cipher.encryptor()
        
        ciphertext = encryptor.update(padded_data) + encryptor.finalize()
        return iv, ciphertext

    def decrypt(self, iv, ciphertext_bytes):
        cipher = Cipher(algorithms.AES(self.key), modes.CBC(iv))
        decryptor = cipher.decryptor()
        
        padded_plaintext = decryptor.update(ciphertext_bytes) + decryptor.finalize()
        
        # Remove PKCS7 padding
        unpadder = padding.PKCS7(128).unpadder()
        try:
            plaintext = unpadder.update(padded_plaintext) + unpadder.finalize()
            return plaintext
        except ValueError:
            print("[!] PADDING ERROR: Invalid PKCS7 padding! Data tampered.")
            return None



In [ ]:
# Testing AES-CBC module
print("--- AES-CBC Padding and Encryption ---")
key = os.urandom(32)  # 256-bit key
cbc_engine = SymmetricCBCModule(key)

secret_msg = b"Sensitive credit card payload data."
iv, ciphertext = cbc_engine.encrypt(secret_msg)

print(f"Plaintext length:  {len(secret_msg)} bytes")
print(f"Ciphertext length: {len(ciphertext)} bytes (Aligned to 16-byte blocks)")

decrypted = cbc_engine.decrypt(iv, ciphertext)
print(f"Decrypted output:  {decrypted.decode()}")



In [ ]:
# 3. Asymmetric RSA Key Validation checks
from cryptography.hazmat.primitives.asymmetric import rsa

def validate_rsa_key_security(private_key_obj):
    """Audits RSA key configuration safety."""
    key_size = private_key_obj.key_size
    print(f"\n--- RSA Key Audit (Size: {key_size} bits) ---")
    if key_size < 2048:
        print("[-] SECURITY WARNING: Weak Key! RSA key sizes below 2048 bits are insecure.")
        return False
    else:
        print("[+] Key configuration passes production safety guidelines.")
        return True

# Generate a weak RSA key (1024 bits - demo only)
weak_key = rsa.generate_private_key(public_exponent=65537, key_size=1024)
validate_rsa_key_security(weak_key)

# Generate a strong key
strong_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
validate_rsa_key_security(strong_key)
